# QLoRA Fine-tuning: Llama 3 8B for Legal Clause Analysis

Fine-tunes Meta Llama 3.1 8B Instruct using QLoRA (4-bit quantization + LoRA) on a legal clause analysis dataset.

**Stack:** HuggingFace Transformers + PEFT + BitsAndBytes (no Unsloth).

**Checkpoint resumption:** Training saves checkpoints to Google Drive. If your Colab runtime disconnects, just re-run all cells — training resumes from the last checkpoint automatically.

**Before running:** `Runtime → Change runtime type → T4 GPU`. Upload `compiled_dataset.jsonl`, `val.jsonl`, `test.jsonl` to your Google Drive under `MyDrive/legal-clauses/`.

## 1. Install Dependencies

In [ ]:
%%capture
!pip install -U transformers peft accelerate bitsandbytes trl datasets

## 2. Load Model (4-bit Quantized)

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"
max_seq_length = 1024

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="sdpa",
)
model.config.use_cache = False

## 3. Add LoRA Adapters

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Prepare Dataset

In [ ]:
import json
from google.colab import drive
from datasets import Dataset

drive.mount('/content/drive')

SYSTEM_PROMPT = """You are a legal clause analyst. Given a legal clause, analyze it and respond with a JSON object containing:
- "type": the clause type
- "summary": a 1-2 sentence summary
- "risk": "Low", "Medium", or "High"
- "reason": a 1-2 sentence risk justification
- "suggestion": a negotiation or improvement suggestion"""

def format_chat(example):
    """Format a single example into Llama 3 chat template."""
    output = example["output"]
    if isinstance(output, str):
        output = json.loads(output)
    output_str = json.dumps(output, indent=2)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["input"]},
        {"role": "assistant", "content": output_str},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}


def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))
    return records


# --- Load data ---
# Update DATA_DIR if your Drive folder is named differently.
DATA_DIR = "/content/drive/MyDrive/legal-clauses"

train_raw = load_jsonl(f"{DATA_DIR}/compiled_dataset.jsonl")
val_raw = load_jsonl(f"{DATA_DIR}/val.jsonl")

train_dataset = Dataset.from_list(train_raw).map(format_chat)
val_dataset = Dataset.from_list(val_raw).map(format_chat)

print(f"Train examples: {len(train_dataset)}")
print(f"Val examples:   {len(val_dataset)}")
print(f"\n--- Sample ---\n{train_dataset[0]['text'][:500]}...")

## 5. Training

Checkpoints are saved to Google Drive every 200 steps. If the runtime disconnects, re-run all cells — training will automatically resume from the last checkpoint.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
import glob

# Save checkpoints to Drive so they survive runtime restarts
OUTPUT_DIR = "/content/drive/MyDrive/legal-clauses/training-outputs"

training_args = TrainingArguments(
    # --- Core ---
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,  # effective batch size = 8
    per_device_eval_batch_size=2,

    # --- Optimizer ---
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=50,
    optim="adamw_8bit",
    lr_scheduler_type="cosine",

    # --- Precision ---
    fp16=True,
    bf16=False,

    # --- Logging & Eval ---
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # --- Misc ---
    seed=42,
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

In [ ]:
# Show memory before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_mem / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name} | {max_memory} GB total | {start_gpu_memory} GB reserved")

In [ ]:
# Auto-resume from last checkpoint if one exists on Drive
checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"))
resume_from = checkpoints[-1] if checkpoints else None

if resume_from:
    print(f"Resuming from checkpoint: {resume_from}")
else:
    print("No checkpoint found — starting fresh")

trainer_stats = trainer.train(resume_from_checkpoint=resume_from)

In [ ]:
# Training summary
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"Training time: {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"Peak GPU memory: {used_memory} GB / {max_memory} GB")
print(f"Final train loss: {trainer_stats.metrics['train_loss']:.4f}")

## 6. Test Inference

In [ ]:
model.eval()

test_clause = """The Borrower shall not, without the prior written consent of the Lender, \
assign, transfer, or delegate any of its rights or obligations under this Agreement \
to any third party. Any attempted assignment without such consent shall be null and void."""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": test_clause},
]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

with torch.no_grad():
    output = model.generate(
        input_ids=inputs,
        max_new_tokens=512,
        temperature=0.3,
        top_p=0.9,
        do_sample=True,
    )

response = tokenizer.decode(output[0][inputs.shape[-1]:], skip_special_tokens=True)
print(response)

## 7. Save Model

Save the LoRA adapters to Drive.

In [ ]:
# Save LoRA adapters to Drive (so they survive runtime shutdown)
SAVE_DIR = "/content/drive/MyDrive/legal-clauses/legal-llama3-qlora"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"LoRA adapters saved to {SAVE_DIR}")

In [ ]:
# Optional: push to Hugging Face Hub
# model.push_to_hub("your-username/legal-llama3-qlora", token="hf_...")
# tokenizer.push_to_hub("your-username/legal-llama3-qlora", token="hf_...")

In [ ]:
# Optional: merge LoRA into base model and save as full model
# merged_model = model.merge_and_unload()
# merged_model.save_pretrained("legal-llama3-merged")
# tokenizer.save_pretrained("legal-llama3-merged")